# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print("Raw response from the model:", result)
    print("---------------------------------------------------------------------")
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

Raw response from the model: {
  "links": [
    {"type": "homepage", "url": "https://edwarddonner.com/"},
    {"type": "about page", "url": "https://edwarddonner.com/about-me-and-about-nebula/"},
    {"type": "curriculum page", "url": "https://edwarddonner.com/curriculum/"},
    {"type": "proficient page", "url": "https://edwarddonner.com/proficient/"},
    {"type": "project page", "url": "https://edwarddonner.com/connect-four/"},
    {"type": "project page", "url": "https://edwarddonner.com/outsmart/"},
    {"type": "blog index", "url": "https://edwarddonner.com/posts/"},
    {"type": "blog post", "url": "https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/"},
    {"type": "blog post", "url": "https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/"},
    {"type": "blog post", "url": "https://edwarddonner.com/2025/11/11/ai-live-event/"},
    {"type": "blog post", "url": "https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'proficient page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog index', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/11/11/ai-live-event/'},
  {'type': 'blog post',
   'url': 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/'},
  {'ty

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 2 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'documentation hub', 'url': 'https://huggingface.co/docs'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
Try HuggingChat Omni – Chat with AI 💬
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Lightricks/LTX-2.3
Updated
6 days ago
•
345k
•
486
Qwen/Qwen3.5-9B
Updated
9 days ago
•
1.39M
•
711
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
4 days ago
•
30.8k
•
392
HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive
Updated
7 days ago
•
127k
•
314
sarvamai/sarvam-105b
Updated
1 day ago
•
4.2k
•
214
Browse 2M+ models
Spaces
Running
on
Zero
186
OBLITERATUS
💥
186
One-click model liberation 

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nTry HuggingChat Omni – Chat with AI 💬\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nLightricks/LTX-2.3\nUpdated\n6 days ago\n•\n345k\n•\n486\nQwen/Qwen3.5-9B\nUpdated\n9 days ago\n•\n1.39M\n•\n712\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\n4 days ago\n•\n30.8k\n•\n392\nHauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive\n

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future. It is a collaboration platform designed specifically for the machine learning community to create, discover, and share models, datasets, and applications. As a central hub for open-source machine learning, Hugging Face empowers engineers, scientists, and AI enthusiasts to build an open and ethical AI future together.

The platform hosts over **2 million models**, **500k+ datasets**, and **1 million+ applications**, providing unparalleled access and collaboration opportunities to users around the globe.

---

## What Hugging Face Offers

- **Models:** Access and share a vast collection of pre-trained machine learning models covering countless AI applications.
- **Datasets:** Explore and collaborate on hundreds of thousands of datasets essential for training and testing AI systems.
- **Spaces:** Deploy and share fully interactive AI-powered applications and demos, making it easy to showcase machine learning models.
- **Buckets:** AI-native object storage designed to meet the demands of machine learning workflows.
- **Open Source Libraries:** Hugging Face supports and fosters open source machine learning tools, facilitating rapid experimentation and innovation.
- **HuggingChat Omni:** An AI chat experience integrated into the platform to interface with AI systems conversationally.

---

## The Hugging Face Community

Hugging Face is much more than a platform—it's a fast-growing, vibrant community constantly advancing open and ethical AI technologies. The users include:

- Machine Learning Engineers
- Data Scientists
- AI Researchers
- Developers building AI applications
- Enthusiasts and students eager to learn and contribute

The community collaborates on improving models, constructing new datasets, and building AI apps, all while sharing knowledge freely.

---

## Company Culture & Values

- **Open & Ethical AI:** Hugging Face is deeply committed to creating AI technologies that are ethical, transparent, and accessible.
- **Collaboration:** The platform thrives on community collaboration and knowledge-sharing to fuel rapid development and innovation.
- **Inclusivity:** Hugging Face empowers diverse users from across the globe to participate in shaping AI’s future.
- **Innovation:** By hosting cutting-edge models and supporting emerging AI technologies, Hugging Face stays at the forefront of the machine learning revolution.
- **Learning & Growth:** Hugging Face fosters a culture where engineers and scientists can learn, grow, and contribute meaningfully to AI advancements.

---

## Careers at Hugging Face

Hugging Face seeks passionate talent eager to build the future of AI alongside a supportive and innovative community. Opportunities include roles in:

- Machine Learning Engineering
- Research and Development
- Software Engineering
- Data Science
- Product Management
- Community Engagement

Working at Hugging Face means contributing to open source, engaging with a leading-edge ML community, and driving real-world AI impact.

Interested candidates can visit the Hugging Face website to explore current openings and join a company transforming how AI is built and shared globally.

---

## Get Involved

- **Explore AI Applications:** Dive into the latest applications powered by Hugging Face technology.
- **Browse Models & Datasets:** Utilize millions of resources for your machine learning projects.
- **Join the Community:** Share your work, collaborate with others, and contribute to the open AI ecosystem.
- **Try HuggingChat Omni:** Experience conversational AI directly integrated into the platform.

---

## Contact & Links

- Website: [huggingface.co](https://huggingface.co)
- Explore models, datasets, and applications online
- Access documentation and pricing plans for enterprise users

---

**Hugging Face – The AI community building the future.**  
Join a movement committed to open, collaborative, and ethical machine learning.

---

*Colors & Branding:*  
Primary colors include bright yellow (#FFD21E), vibrant orange (#FF9D00), and neutral gray (#6B7280), reflecting the vibrant and dynamic spirit of the community.  

---

**Come create, discover, and innovate with Hugging Face.**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    print("Streaming brochure content:", stream)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Hugging Face: Building the Future of AI Together

---

## About Hugging Face

Hugging Face is the collaboration platform at the heart of the AI and machine learning revolution. Serving as a central hub for the worldwide machine learning community, Hugging Face empowers engineers, scientists, researchers, and AI enthusiasts to create, share, explore, and experiment with open-source machine learning models, datasets, and applications. Their mission is to build an open and ethical AI future by connecting a fast-growing community with the most popular open-source ML libraries and cutting-edge AI research.

---

## Platform Highlights

- **Models:** Access and contribute to a massive repository of over 2 million machine learning models, spanning diverse AI tasks.
- **Datasets:** Browse from more than 500,000 curated datasets to train and evaluate machine learning models.
- **Spaces:** Host and share AI applications powered by models in an interactive environment.
- **Buckets:** AI-native object storage designed for robust and scalable data handling.
- **HuggingChat Omni:** A new conversational AI tool to chat with AI instantly.
- **Collaboration:** Unlimited hosting and real-time collaboration on ML projects supporting faster innovation.

---

## Customers & Enterprise Solutions

Hugging Face offers tailored solutions for teams and enterprises wanting to scale AI efforts securely and efficiently:

- **Team Plans:** Starting at $20/user/month — ideal for SMEs and growing teams.
- **Enterprise Plans:** Flexible contracts with features including:
  - Enterprise-grade security with Single Sign-On (SSO).
  - Granular access controls and resource group management.
  - Audit logging and centralized token/token approval policy management.
  - Private storage and dataset viewers.
  - Enhanced compute options with ZeroGPU quota boost for greater scalability.
  - Analytics dashboards for usage and spending insights.

These tools ensure organizations maintain control over their AI workflows while boosting productivity.

---

## Company Culture

Hugging Face fosters an open, collaborative, and ethical AI community. With a strong emphasis on transparency and inclusivity, the company empowers talents ranging from novice enthusiasts to world-leading AI scientists. The culture encourages sharing knowledge, pushing technological boundaries, and working collectively to develop responsible and cutting-edge AI technologies.

---

## Careers at Hugging Face

Joining Hugging Face means becoming part of a passionate team shaping the future of AI and machine learning. The company seeks individuals eager to contribute to open source, innovate with state-of-the-art technologies, and enhance global AI accessibility.

Available roles typically include:

- Machine Learning Engineers & Scientists
- Software Engineers
- Community Managers
- Research Scientists
- Data Engineers

Hugging Face promotes a collaborative, remote-friendly environment valuing curiosity, ownership, and impact-driven work.

Explore current openings and apply at their [Careers page](https://huggingface.co/careers).

---

## Connect & Learn

- Visit the Hugging Face Hub to explore models, datasets, and AI applications: https://huggingface.co
- Join their community on GitHub, Twitter, LinkedIn, and Discord
- Access comprehensive documentation, tutorials, and forums to grow your AI skills

---

## Brand Identity

- **Primary Colors:** Yellow (#FFD21E), Orange (#FF9D00), Gray (#6B7280)
- **Logo & Assets:** Available in multiple formats (.svg, .png, .ai) for consistent branding use

---

**Hugging Face — The AI community building the future.**

Start collaborating today and accelerate your machine learning projects with the world’s leading open-source AI platform.

Streaming brochure content: <openai.Stream object at 0x1134ed2b0>


In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>